In [1]:
import torch
import dnnlib
import legacy

In [3]:
network_pkl = 'ffhq-256.pkl'
device = torch.device('cuda')
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

In [4]:
print(G)

Generator(
  (synthesis): SynthesisNetwork(
    w_dim=512, num_ws=14, img_resolution=256, img_channels=3, num_fp16_res=4
    (b4): SynthesisBlock(
      resolution=4, architecture=skip
      (conv1): SynthesisLayer(
        in_channels=512, out_channels=512, w_dim=512, resolution=4, up=1, activation=lrelu
        (affine): FullyConnectedLayer(in_features=512, out_features=512, activation=linear)
      )
      (torgb): ToRGBLayer(
        in_channels=512, out_channels=3, w_dim=512
        (affine): FullyConnectedLayer(in_features=512, out_features=512, activation=linear)
      )
    )
    (b8): SynthesisBlock(
      resolution=8, architecture=skip
      (conv0): SynthesisLayer(
        in_channels=512, out_channels=512, w_dim=512, resolution=8, up=2, activation=lrelu
        (affine): FullyConnectedLayer(in_features=512, out_features=512, activation=linear)
      )
      (conv1): SynthesisLayer(
        in_channels=512, out_channels=512, w_dim=512, resolution=8, up=1, activation=lrelu
 

In [5]:
affines = []

for name, module in G.synthesis.named_modules():
    if hasattr(module, "affine"):
        affines.append(module.affine)
print(len(affines))
print(affines)

20
[FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=512, activation=linear), FullyConnectedLayer(in_features=512, out_features=256, activation=linear), FullyConnectedLayer(i

In [6]:
print(affines[0].weight.shape)

torch.Size([512, 512])


In [7]:
mapping = G.mapping

In [8]:
z = torch.randn(1, 512).to(device)
ws = G.mapping(z, None)
print(ws.shape)

torch.Size([1, 14, 512])


In [18]:
class StyleAffineMapper(torch.nn.Module):
    def __init__(self, mapping, affines):
        super().__init__()
        self.mapping = mapping
        self.affines = torch.nn.ModuleList(affines)

    def forward(self, z):
        w = self.mapping(z, None) # shape [batch, 14, 512]
        outputs = []
        
        # Handle First Block
        outputs.append(self.affines[0](w[:, 0])) # conv1
        outputs.append(self.affines[1](w[:, 1])) # toRGB
        
        # Rest of the blocks
        w_idx = 2  
        affine_idx = 2
        for _ in range(6):
            # conv0
            outputs.append(self.affines[affine_idx](w[:, w_idx]))
            # conv1
            outputs.append(self.affines[affine_idx + 1](w[:, w_idx + 1]))
            # toRGB (reuses the second w vector of this block)
            outputs.append(self.affines[affine_idx + 2](w[:, w_idx + 1]))
            
            w_idx += 2
            affine_idx += 3
        
        return outputs

In [ ]:
style_generator = StyleAffineMapper(mapping, affines)
# print(style_generator)
z = torch.randn(1, 512).to(device)
style_outputs = style_generator(z)
for i, style_vector in enumerate(style_outputs):
    print(f"Style vector {i+1}: shape {style_vector.shape}")

Style vector 0: shape torch.Size([1, 512])
Style vector 1: shape torch.Size([1, 512])
Style vector 2: shape torch.Size([1, 512])
Style vector 3: shape torch.Size([1, 512])
Style vector 4: shape torch.Size([1, 512])
Style vector 5: shape torch.Size([1, 512])
Style vector 6: shape torch.Size([1, 512])
Style vector 7: shape torch.Size([1, 512])
Style vector 8: shape torch.Size([1, 512])
Style vector 9: shape torch.Size([1, 512])
Style vector 10: shape torch.Size([1, 512])
Style vector 11: shape torch.Size([1, 512])
Style vector 12: shape torch.Size([1, 256])
Style vector 13: shape torch.Size([1, 256])
Style vector 14: shape torch.Size([1, 256])
Style vector 15: shape torch.Size([1, 128])
Style vector 16: shape torch.Size([1, 128])
Style vector 17: shape torch.Size([1, 128])
Style vector 18: shape torch.Size([1, 64])
Style vector 19: shape torch.Size([1, 64])
